# Task 3 Demo - Anomaly Detection Methods and Top-50 Selection

In this notebook I demonstrate anomaly detection with multiple methods and deterministic top-50 output.

What I show:

1. Build anomaly-focused features.
2. Compare Isolation Forest, LOF, and distance-based kNN scoring.
3. Analyze overlap between methods.
4. Export exactly 50 anomaly IDs in assignment format.

Reader guide:

- This notebook is the final multi-method Task-3 comparison.
- All rankings and overlap tables are exported to `data/results/`.
- The final `anomalies.csv` is deterministic and contains exactly 50 IDs.

In [8]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors

from core.data_io import ArticleDataset
from preprocessing import TextNormalizer, TextPreprocessor
from exploration import attach_original_text_by_doc_id

## Load data and build anomaly feature spaces

I use anomaly-focused normalization and dense TF-IDF+LSA features to capture both structure and semantics.

In [9]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

document_ids = articles_data_frame["doc_id"].tolist()
raw_texts = articles_data_frame["text"].tolist()

text_normalizer = TextNormalizer()
normalized_bundle = text_normalizer.normalize_for_both_tasks(raw_texts)

anomaly_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
    dense_embedding_dimension=256,
    random_seed=42,
)
anomaly_feature_matrix = anomaly_preprocessor.fit_transform(normalized_bundle.anomaly_texts)

anomaly_feature_matrix.shape

(2164, 256)

## Method 1 - Isolation Forest

Isolation Forest is used as the main ranking method.
Lower `score_samples` values indicate stronger anomaly behavior.

In [10]:
isolation_forest_model = IsolationForest(contamination=0.02, random_state=42)
isolation_forest_model.fit(anomaly_feature_matrix)

# Lower score means more anomalous for score_samples.
isolation_forest_scores = isolation_forest_model.score_samples(anomaly_feature_matrix)

isolation_forest_ranked = (
    pd.DataFrame({"doc_id": document_ids, "score": isolation_forest_scores})
    .sort_values("score", ascending=True)
    .reset_index(drop=True)
)
isolation_forest_ranked["rank"] = isolation_forest_ranked.index + 1
isolation_forest_ranked_with_text = attach_original_text_by_doc_id(
    anomaly_table=isolation_forest_ranked,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
isolation_forest_ranked_with_text.to_csv(results_data_directory_path / "notebook_07_isolation_forest_ranking.csv", index=False)

isolation_forest_ranked_with_text.head(20)

,doc_id,score,rank,text
0,DOC_00443,-0.487344,1,Am I justified in being pissed off at this doc...
1,DOC_00909,-0.486837,2,Les Bartel's comments: Let me add my .02 in. I...
2,DOC_01833,-0.485884,3,: Am I justified in being pissed off at this d...
3,DOC_01603,-0.485483,4,Some recent postings remind me that I had read...
4,DOC_00223,-0.483222,5,Help!! I need code/package/whatever to take 3-...
5,DOC_00718,-0.482727,6,+At one time there was speculation that the fi...
6,DOC_00192,-0.482307,7,+++ ++Once inflated the substance was no longe...
7,DOC_01025,-0.480305,8,I need as much information about Cosmos 2238 a...
8,DOC_02026,-0.479968,9,Does anyone know ifthe STS-56 email press kit ...
9,DOC_00085,-0.479878,10,At one time there was speculation that the fir...


## Method 2 - Local Outlier Factor (LOF)

LOF adds a local-density perspective.
More negative LOF values indicate stronger outlier behavior.

In [11]:
lof_model = LocalOutlierFactor(n_neighbors=35, contamination=0.02)
lof_predictions = lof_model.fit_predict(anomaly_feature_matrix)

# LOF stores negative factors; lower (more negative) means more anomalous.
lof_raw_factors = lof_model.negative_outlier_factor_
lof_ranked = pd.DataFrame(
    {"doc_id": document_ids, "score": lof_raw_factors, "predicted_outlier": lof_predictions == -1}
)
lof_ranked = lof_ranked.sort_values("score", ascending=True).reset_index(drop=True)
lof_ranked["rank"] = lof_ranked.index + 1
lof_ranked_with_text = attach_original_text_by_doc_id(
    anomaly_table=lof_ranked,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
lof_ranked_with_text.to_csv(results_data_directory_path / "notebook_07_lof_ranking.csv", index=False)

lof_ranked_with_text.head(20)

,doc_id,score,predicted_outlier,rank,text
0,DOC_00098,-1.438458,True,1,GAME(S) OF 4/15 --------------- ADIRONDACK 6 C...
1,DOC_00089,-1.432247,True,2,AHL CALDER CUP PLAYOFF GAME(S) PLAYED ON 4/16 ...
2,DOC_00155,-1.412230,True,3,1993 CALDER CUP PLAYOFF SCHEDULE AND RESULTS h...
3,DOC_00464,-1.405302,True,4,Punchline #3: it would be a good idea just to ...
4,DOC_00030,-1.375277,True,5,Nick Haines sez; Level 5? Out of how many? Wha...
5,DOC_01449,-1.361371,True,6,<>Hismanal (astemizole) is most definitely lin...
6,DOC_00001,-1.361125,True,7,"When I first saw this, I thought for a second ..."
7,DOC_00711,-1.348279,True,8,"Pat sez; Yeah, but a windscreen cut down most ..."
8,DOC_01894,-1.320184,True,9,"Concerning the proposed newsgroup split, I per..."
9,DOC_00626,-1.319915,True,10,": Concerning the proposed newsgroup split, I p..."


## Method 3 - Distance-based kNN score

This method scores each document by mean cosine distance to neighbors.
Higher distance indicates a less typical local neighborhood.

In [12]:
neighbor_model = NearestNeighbors(n_neighbors=20, metric="cosine")
neighbor_model.fit(anomaly_feature_matrix)

neighbor_distances, _ = neighbor_model.kneighbors(anomaly_feature_matrix)
# Higher average distance means less similar neighborhood.
knn_distance_scores = neighbor_distances.mean(axis=1)

knn_ranked = (
    pd.DataFrame({"doc_id": document_ids, "score": knn_distance_scores})
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)
knn_ranked["rank"] = knn_ranked.index + 1
knn_ranked_with_text = attach_original_text_by_doc_id(
    anomaly_table=knn_ranked,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
knn_ranked_with_text.to_csv(results_data_directory_path / "notebook_07_knn_distance_ranking.csv", index=False)

knn_ranked_with_text.head(20)

,doc_id,score,rank,text
0,DOC_00107,0.688190,1,"I would guess that it requires X, almost certa..."
1,DOC_00476,0.687391,2,"As an additional data point, I have run Castro..."
2,DOC_02022,0.687356,3,"Okay, I've received a whole lot of requests fo..."
3,DOC_01248,0.686094,4,Sheesh! I don't know what kind of women they h...
4,DOC_00392,0.685650,5,How 'bout some more info on that alleged super...
5,DOC_01075,0.681636,6,We have a minivas-2 and we want to record to a...
6,DOC_00510,0.680495,7,"Such submissions have been made before, e.g. r..."
7,DOC_00163,0.678914,8,"Sorry to put a damper on your plans, but I was..."
8,DOC_00374,0.676638,9,I have a friend who has a very pronounced slou...
9,DOC_00572,0.676085,10,What is the deal with life on Mars? I save the...


## Compare overlaps between top-50 sets

Overlap helps show where methods agree or disagree on likely anomalies.

In [13]:
def top_k_ids(ranked_table: pd.DataFrame, top_k: int = 50) -> set[str]:
    """Returns top-k ids from a ranked anomaly table."""
    return set(ranked_table.head(top_k)["doc_id"].astype(str).tolist())


if_top_50 = top_k_ids(isolation_forest_ranked, 50)
lof_top_50 = top_k_ids(lof_ranked, 50)
knn_top_50 = top_k_ids(knn_ranked, 50)

overlap_table = pd.DataFrame(
    [
        {"pair": "if_vs_lof", "overlap": len(if_top_50.intersection(lof_top_50))},
        {"pair": "if_vs_knn", "overlap": len(if_top_50.intersection(knn_top_50))},
        {"pair": "lof_vs_knn", "overlap": len(lof_top_50.intersection(knn_top_50))},
    ]
)
overlap_table.to_csv(results_data_directory_path / "notebook_07_top50_overlap_table.csv", index=False)
overlap_table

,pair,overlap
0,if_vs_lof,22
1,if_vs_knn,2
2,lof_vs_knn,2


## Export assignment-formatted anomalies.csv (exactly 50 ids)

I use Isolation Forest ranking as the deterministic final selector.

In [14]:
final_top_50 = isolation_forest_ranked.head(50).copy()
final_top_50["anomaly"] = final_top_50.index + 1
anomaly_submission_data_frame = final_top_50[["anomaly", "doc_id"]]
anomaly_submission_with_text_data_frame = attach_original_text_by_doc_id(
    anomaly_table=anomaly_submission_data_frame,
    document_ids=document_ids,
    raw_texts=raw_texts,
)

# This file matches the assignment output format.
anomaly_submission_with_text_data_frame.to_csv(results_data_directory_path / "anomalies.csv", index=False)
anomaly_submission_with_text_data_frame.to_csv(results_data_directory_path / "notebook_07_anomalies_candidate.csv", index=False)

anomaly_submission_with_text_data_frame.head()

,anomaly,doc_id,text
0,1,DOC_00443,Am I justified in being pissed off at this doc...
1,2,DOC_00909,Les Bartel's comments: Let me add my .02 in. I...
2,3,DOC_01833,: Am I justified in being pissed off at this d...
3,4,DOC_01603,Some recent postings remind me that I had read...
4,5,DOC_00223,Help!! I need code/package/whatever to take 3-...


### Files produced by this notebook

- `data/results/notebook_07_isolation_forest_ranking.csv`
- `data/results/notebook_07_lof_ranking.csv`
- `data/results/notebook_07_knn_distance_ranking.csv`
- `data/results/notebook_07_top50_overlap_table.csv`
- `data/results/notebook_07_anomalies_candidate.csv`
- `data/results/anomalies.csv`
